# Notebook 02: CT-RATE Streaming & 3D ResNet Contrastive Pretraining

This notebook walks through:
1. Disjoint patient splitting with mathematical assertion of zero split leakage.
2. Single-scan streaming pipeline with immediate disk deletion to prevent disk exhaustion.
3. 3D volumetric patch extraction with >=70% lung parenchyma containment.
4. 3D ResNet encoder and NT-Xent contrastive loss optimization.


In [ ]:
import torch
import pandas as pd
from copd_anomaly.data.ctrate_streamer import create_tiered_splits, verify_split_leakage
from copd_anomaly.models.encoder import ResNet3DEncoder
from copd_anomaly.models.losses import NTXentLoss
from copd_anomaly.models.augmentations import ContrastiveDualViewTransform

print('Modules loaded successfully. CUDA available:', torch.cuda.is_available())


## 1. Zero-Leakage Patient Splitting
Validates that ctrate_train, ctrate_val, and ctrate_test share strictly zero patients.


In [ ]:
# Synthetic demo metadata
train_meta = pd.DataFrame([{'VolumeName': f'train_{i:04d}_a_1.nii.gz'} for i in range(150)])
valid_meta = pd.DataFrame([{'VolumeName': f'valid_{i:04d}_a_1.nii.gz'} for i in range(80)])

splits = create_tiered_splits(train_meta, valid_meta, tier='lean', seed=42)
print('Splits successfully generated:')
print('  Train subjects:', len(splits['train']))
print('  Val subjects:  ', len(splits['val']))
print('  Test subjects: ', len(splits['test']))


## 2. 3D ResNet Architecture & Contrastive Forward Pass
Each 3D patch (50x50x50) produces a latent representation $z_{\text{emb}} \in \mathbb{R}^{128}$ and normalized projection $z_{\text{proj}} \in \mathbb{R}^{64}$.


In [ ]:
model = ResNet3DEncoder(in_channels=1, base_filters=16, embedding_dim=128, projection_dim=64)
x_dummy = torch.rand(4, 1, 32, 32, 32)  # Batch of 4 patches

dual_transform = ContrastiveDualViewTransform()
v1, v2 = dual_transform(x_dummy)

z1_emb, z1_proj = model(v1)
z2_emb, z2_proj = model(v2)

criterion = NTXentLoss(temperature=0.07)
loss = criterion(z1_proj, z2_proj)
print(f'Contrastive NT-Xent Loss: {loss.item():.4f}')
